# Planning-Lab Lesson 1: Search Strategies

In this first lesson we will work both on Search Strategies. 

### Maze Environments
The environments used is **SmallMaze** (visible in the figure).

<img src="images/maze.png" width="300">

The agent starts in cell $(0, 2)$ and has to reach the treasure in $(4, 3)$.

In order to use the environment we need first to import the packages of OpenAI Gym. Notice that due to the structure of this repository, we need to add the parent directory to the path

In [1]:
import os, sys, time, math

module_path = os.path.abspath(os.path.join('../tools'))
if module_path not in sys.path:
    sys.path.append(module_path)

from utils.ai_lab_functions import *
import gym, envs

# Uninformed search
### Assignment 1: Breadth-First Search (BFS)

**Your first assignment is to implement the BFS algorithm on SmallMaze. In particular, given the *tree search* version of BFS, you should implement the *graph search* version.**

The results returned by your **BFS** must be in the following form (path, time_cost, space_cost), more in detail:

- **path** - a tuple of state identifiers forming a path from the start state to the goal state. None if no solution is found.
- **time_cost** - the number of nodes checked during the exploration.
- **space_cost** - the maximum number of nodes in memory simultaneously.

**For the time_cost, we consider a node checked after its generation!**

After the correctness of your implementations has been assessed, you can run the algorithms on the **SmallMaze** environment.

Functions to implement:

- BFS_GraphSearch(problem)

Function **build_path(node)** can be used to return a tuple of states from the root node (excluded) to another node by following parent links.

Here is the pseudo-code from the book **Artificial Intelligence: A Modern Approach** for *Graph Search* and *Tree Search*:

<img src="images/tree-graph-search.png" width="600">

Here is the pseudo-code from the book **Artificial Intelligence: A Modern Approach** for the *BFS* algorithm, note that it refers to the implementation of the *Graph Search Version*:

<img src="images/BFS2.png" width="600">

**Here's an implementation of the BFS algorithm in its Tree Search version:**

In [ ]:
def BFS_TreeSearch(problem):
    """
    Tree Search BFS
    
    Args:
        problem: OpenAI Gym environment
        
    Returns:
        (path, time_cost, space_cost): solution as a path and stats.
    """
    #Initialization
    node = Node(problem.startstate, None)
    time_cost, space_cost = 1,1
    
    #Check if the starting state is already the target
    if node.state == problem.goalstate:
        return build_path(node), time_cost, space_cost
    
    #Queue for managing the search frontier
    frontier = NodeQueue()
    frontier.add(node)    
    
    while not frontier.is_empty():
        #Expand the next node in the queue
        node = frontier.remove()
        
        #Iterate through all available actions in the environment
        for action in range(problem.action_space.n):
            #Generate a successor node
            child = Node(problem.sample(node.state, action), node)
            time_cost += 1
            
            # Goal test: verify if the child reaches the objective
            if child.state == problem.goalstate: 
                return build_path(child), time_cost, space_cost
            
            frontier.add(child)
        
        #Monitor the peak size of the frontier for space complexity
        if len(frontier) > space_cost:
            space_cost = len(frontier)
    
    return None, time_cost, space_cost

**Now you can implement the same BFS algorithm in the Graph Search version:**

In [ ]:
def BFS_GraphSearch(problem):
    """
    Graph Search BFS
    
    Args:
        problem: OpenAI Gym environment
        
    Returns:
        (path, time_cost, space_cost): solution as a path and stats.
    """
    #Initialization
    node = Node(problem.startstate, None)
    time_cost, space_cost = 1, 1
    
    #Check if the starting state is already the target
    if node.state == problem.goalstate:
        return build_path(node), time_cost, space_cost
    
    #Setup frontier (queue) and the set of visited states
    frontier = NodeQueue()
    explored = set()
    frontier.add(node)
    
    while not frontier.is_empty():
        #Pop the shallowest node from the frontier
        node = frontier.remove()
        explored.add(node.state)

        #Iterate over all possible discrete actions        
        for action in range(problem.action_space.n):
            #Create a child node by sampling the environment
            child = Node(problem.sample(node.state, action), node)
            time_cost += 1
            
            #Verify if the state has already been seen or is queued
            if child.state not in explored and child.state not in frontier:
                #Goal reached: return path and current metrics
                if child.state == problem.goalstate: 
                    return build_path(child), time_cost, space_cost
                frontier.add(child)
        
        #Track the peak memory footprint (frontier + explored set)
        current_memory = len(frontier) + len(explored)
        if current_memory > space_cost:
            space_cost = current_memory 

    return None, time_cost, space_cost

**The following code calls your graph search version and the tree search version of BFS and checks the results:**

In [4]:
envname = "SmallMaze-v0"
#envname = "GrdMaze-v0"
#envname = "BlockedMaze-v0"
environment = gym.make(envname)

solution_ts, time_ts, memory_ts = BFS_TreeSearch(environment)
solution_gs, time_gs, memory_gs = BFS_GraphSearch(environment)

results = CheckResult_L1A1([solution_ts, time_ts, memory_ts], [solution_gs, time_gs, memory_gs], environment)
results.check_sol_ts()
results.check_sol_gs()

##########################################
#######  BFS TREE SEARCH PROBLEM  ########
##########################################
Solution: [(0, 1), (0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (4, 1), (4, 2), (4, 3)]
N° of nodes explored: 103723
Max n° of nodes in memory: 77791

##########################################
#######  BFS Graph SEARCH PROBLEM  #######
##########################################
Your solution: [(0, 1), (0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (4, 1), (4, 2), (4, 3)]
N° of nodes explored: 59
Max n° of nodes in memory: 15

===> Your solution is correct!



## Assignment 2:  Depth-Limited Search (DLS) and Iterative Deepening depth-first Search (IDS)

**Your second assignment is to implement the IDS algorithm on SmallMaze. 
In particular, given Deph Limited Search (*DLS*) in the *tree search* version, you are required to implement the Ierative Depening Search (*IDS*) function.**

Similarly to assignment 1, the results returned by your ids must be in the following form (path, Time Cost, Space Cost) described above. After the correctness of your implementations has been assessed, you can run the algorithms on the **SmallMaze** environment.

Functions to implement:

- IDS(problem)

Function **build_path(node)** can be used to return a tuple of states from the root node (excluded) to another node by following parent links.

Here is the pseudo-code from the book **Artificial Intelligence: A Modern Approach** for the *Depth-Limited Search (Tree Search Version)* and *Iterative deepening depth-first search (Tree Search Version)*:

<img src="images/dls.png" width="600">

<img src="images/ids.png" width="600">

Note that Node() has a depthcost variable that represents the depth of the node in the search tree. This variable is automatically set by the Node constructor: if the root node has a depthcost of 0, its children will have a depthcost increased by 1. The depthcost is useful to compute the space cost for IDS. An example is shown below:

In [5]:

start = environment.startstate
root = Node(start)  # parent = None and depthcost = 0 as default
child = Node(environment.sample(start, 0), root) # the depthcost is set automatically in the Node constructor
print("Root depthcost: {}\tChild depthcost: {}".format(root.depthcost, child.depthcost))

Root depthcost: 0	Child depthcost: 1


In [6]:
def DLS(problem, limit, RDLS_Function):
    """
    DLS
    
    Args:
        problem: OpenAI Gym environment
        limit: depth limit for the exploration, negative number means 'no limit'
        
    Returns:
        (path, time_cost, space_cost): solution as a path and stats.
    """
        
    node = Node(problem.startstate, None)
    return RDLS_Function(node, problem, limit, set())

In [7]:
def Recursive_DLS_TreeSearch(node, problem, limit, explored=None):
    """
    DLS (Tree Search Version)
    
    Args:
        node: node to explore
        problem: OpenAI Gym environment
        limit: depth limit for the exploration, negative number means 'no limit'
        
    Returns:
        (path, time_cost, space_cost): solution as a path and stats.
    """
    
    #Base case: Goal reached
    if node.state == problem.goalstate:
        return build_path(node), 1, node.depthcost
    
    #Base case: Depth limit reached (cutoff)
    if limit == 0:
        return "cut_off", 1, node.depthcost
    
    #Initialization
    time_cost = 1 
    space_cost = node.depthcost
    cut_off_occurred = False

    #Expansion of the current node's children
    for action in range(problem.action_space.n):
        #Generate successor node via environmental sampling
        child = Node(problem.sample(node.state, action), node)

        #Recursive exploration with decreased depth limit
        result = Recursive_DLS_TreeSearch(child, problem, limit-1)

        #Accumulate time and update peak space cost
        time_cost += result[1]
        if result[2] > space_cost:
            space_cost = result[2]

        #Process recursion result        
        if result[0] == "cut_off":
            cut_off_occurred = True
        elif result[0] != "failure":
            #If it's not failure and not cutoff, we found the goal
            return result[0], time_cost, space_cost
    
    #Determine final return value based on cutoff flag
    if(cut_off_occurred):
        return "cut_off", time_cost, space_cost
    
    return "failure", time_cost, space_cost

**Study the provided code for Depth Limited Search (DLS) and Recursive_DLS_TreeSearch and implement the Iterative Deepening Search function reported below.**

In [ ]:
def IDS(problem, DLS_Function):
    """
    Iteartive_DLS DLS
    
    Args:
        problem: OpenAI Gym environment
        
    Returns:
        (path, time_cost, space_cost): solution as a path and stats.
    """
    #Initialization
    node = Node(problem.startstate, None)
    start = Node(problem.startstate)
    total_time_cost, total_space_cost = 0, 1

    #Iteratively explore from depth 0 upwards
    for depth in zero_to_infinity():
        #Perform Depth-Limited Search at current depth limit
        result = DLS(problem, depth, DLS_Function)

        #Aggregated performance metrics
        total_time_cost += result[1]
        if result[2] > total_space_cost:
            total_space_cost = result[2]
        
        #Evaluation of the search outcome
        cut_off_occured = result[0] == "cut_off"
        
        #If no cutoff happened, we found a solution or reached a dead end (failure)
        if not cut_off_occured:
            return result[0], total_time_cost, total_space_cost, depth

    return None, total_time_cost, total_space_cost, 0

The following code calls your version of IDS and checks the results:

In [9]:
envname = "SmallMaze-v0"
#envname = "GrdMaze-v0"
#envname = "BlockedMaze-v0"
environment = gym.make(envname)

solution_ts, time_ts, memory_ts, iterations_ts = IDS(environment, Recursive_DLS_TreeSearch)

#=== Recursive DLS Tree Search ===
#start = environment.startstate
#root = Node(start)

# imposta una profondità limite (puoi regolarla in base al problema)
#limit = 7  

#solution_ts, time_ts, memory_ts= Recursive_DLS_TreeSearch(root, environment, limit)

results = CheckResult_L1A2([solution_ts, time_ts, memory_ts, iterations_ts], environment)
results.check_sol_ts()

##########################################
#######  IDS TREE SEARCH PROBLEM  ########
##########################################
Necessary Iterations: 9
Your solution: [(0, 1), (0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (4, 1), (4, 2), (4, 3)]
N° of nodes explored: 138298
Max n° of nodes in memory: 9

===> Your solution is correct!



#### Discussion

Now that you have correctly implemented both BFS and IDS what can you say about the solutions they compute? Are there significant differences in the stats?